<a href="https://www.kaggle.com/code/felixoctavius/eda-cleaning-online-retail-ii?scriptVersionId=327401048" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Gabung Data 2 Tahun

In [2]:
def load_and_combine_data(file_path_1, file_path_2):
    print("Loading and combining datasets...")
    df1 = pd.read_csv(file_path_1, encoding='latin1')
    df2 = pd.read_csv(file_path_2, encoding='latin1')

    # Bersihkan BOM untuk mencegah kesalah teknis
    df1.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    df2.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    
    # Gabung data 2 tahun
    df_combined = pd.concat([df1, df2], ignore_index=True)
    print(f"Combined dataset shape: {df_combined.shape[0]:,} rows.")
    return df_combined

In [3]:
path_2009 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2009_2010.csv'
path_2010 = '/kaggle/input/datasets/felixoctavius/online-retail-ii/online_retail_II_2010_2011.csv'

In [4]:
df_raw = load_and_combine_data(path_2009, path_2010)

Loading and combining datasets...
Combined dataset shape: 1,067,371 rows.


In [5]:
df_raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/2009 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/2009 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/2009 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/2009 7:45,1.25,13085.0,United Kingdom


# EDA Data Mentah

In [6]:
df_raw.info()
print(f"Total baris: {df_raw.shape[0]} | Total kolom: {df_raw.shape[1]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
Total baris: 1067371 | Total kolom: 8


In [7]:
# Mengubah hasil hitungan menjadi DataFrame agar rapi
country_stats = df_raw['Country'].value_counts().reset_index()
country_stats.columns = ['Country', 'Jumlah_Baris']

# Menghitung persentase kontribusi setiap negara
country_stats['Persentase (%)'] = (country_stats['Jumlah_Baris'] / len(df_raw)) * 100

# Menampilkan 10 besar negara
print("--- 10 NEGARA DENGAN BARIS DATA TERBANYAK ---")
print(country_stats.head(10).to_string(index=False))

--- 10 NEGARA DENGAN BARIS DATA TERBANYAK ---
       Country  Jumlah_Baris  Persentase (%)
United Kingdom        981330       91.938979
          EIRE         17866        1.673832
       Germany         17624        1.651160
        France         14330        1.342551
   Netherlands          5140        0.481557
         Spain          3811        0.357045
   Switzerland          3189        0.298771
       Belgium          3123        0.292588
      Portugal          2620        0.245463
     Australia          1913        0.179225


In [8]:
# Menghitung jumlah Invoice unik per negara
tx_per_country = df_raw.groupby('Country')['Invoice'].nunique().reset_index()
tx_per_country.columns = ['Country', 'Jumlah_Transaksi']

# Mengurutkan dari transaksi terbanyak
tx_per_country = tx_per_country.sort_values(by='Jumlah_Transaksi', ascending=False)

# Menghitung persentase kontribusi transaksi
total_tx = tx_per_country['Jumlah_Transaksi'].sum()
tx_per_country['Persentase (%)'] = (tx_per_country['Jumlah_Transaksi'] / total_tx) * 100

print(f"Total seluruh transaksi unik: {total_tx:,}\n")
print("--- 10 NEGARA DENGAN TRANSAKSI TERBANYAK ---")
print(tx_per_country.head(10).to_string(index=False))

Total seluruh transaksi unik: 53,628

--- 10 NEGARA DENGAN TRANSAKSI TERBANYAK ---
       Country  Jumlah_Transaksi  Persentase (%)
United Kingdom             49108       91.571567
       Germany              1095        2.041844
          EIRE               806        1.502946
        France               746        1.391064
   Netherlands               250        0.466174
         Spain               188        0.350563
       Belgium               183        0.341240
        Sweden               129        0.240546
      Portugal               124        0.231222
   Switzerland               123        0.229358


In [9]:
missing_data = df_raw.isnull().sum()
missing_percent = (df_raw.isnull().sum() / len(df_raw)) * 100

In [10]:
missing_df = pd.DataFrame({'Jumlah Missing': missing_data, 'Persentase (%)': missing_percent})
print(missing_df[missing_df['Jumlah Missing'] > 0])

             Jumlah Missing  Persentase (%)
Description            4382        0.410541
Customer ID          243007       22.766873


In [11]:
total_duplikat = df_raw.duplicated().sum()
print(f"Jumlah transaksi duplikat: {total_duplikat} ({total_duplikat / len(df_raw) * 100:.2f}%)")

if total_duplikat > 0:
    print(df_raw[df_raw.duplicated(keep=False)].head(10).sort_values(by='Description'))

Jumlah transaksi duplikat: 34335 (3.22%)
    Invoice StockCode                        Description  Quantity  \
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
383  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
363  489517     21912           VINTAGE SNAKES & LADDERS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   

         InvoiceDate  Price  Customer ID         Country  
365  12/1/2009 11:34   3.75      16329.0  United Kingdom  
367  12/1/2009 11:34   0.65      16329.0  United Kingdom  
384  12/1/2

In [12]:
# Ditemukan nilai quantity dan price yang negatif
df_raw[['Quantity', 'Price']].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


# Data Cleaning

In [13]:
def clean_retail_pipeline(df):
    print("\nStarting basic cleaning...")
    df_clean = df.copy()

    # 0. Filter by country (United Kingdom)
    df_clean = df_clean[df_clean['Country'] == 'United Kingdom']
    df_clean.drop(columns=['Country'], inplace=True)
    
    # 1. Handle Column Name Variations
    col_customer = 'Customer ID' if 'Customer ID' in df_clean.columns else 'CustomerID'
    col_price = 'Price' if 'Price' in df_clean.columns else 'UnitPrice'
    
    # 2. Drop Exact Duplicates & Missing Values
    df_clean.drop_duplicates(inplace=True)
    df_clean.dropna(subset=[col_customer, 'Description', 'Invoice'], inplace=True)
    
    # 3. Format Data Types
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
    # Convert Customer ID to string without the .0 float formatting
    df_clean[col_customer] = df_clean[col_customer].astype(int).astype(str)
    # Standardize string formatting for Description
    df_clean['Description'] = df_clean['Description'].astype(str).str.upper().str.strip()
    
    # 4. Filter Anomalies (Returns, Free Items, System Codes)
    df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean[col_price] > 0)]
    
    print("\nStarting advanced product standardization (Latest Transaction Method)...")
    
    # 5. Master Data Management: Standardize Descriptions to Latest Entry
    # Sort chronologically
    df_clean = df_clean.sort_values(by='InvoiceDate', ascending=True)
    
    # Extract the most recent description for every StockCode
    latest_descriptions = df_clean.drop_duplicates(subset=['StockCode'], keep='last')
    description_mapping = latest_descriptions.set_index('StockCode')['Description'].to_dict()
    
    # Map the latest description back to all historical transactions
    df_clean['Description'] = df_clean['StockCode'].map(description_mapping)
    
    # Remove BOM
    df_clean.rename(columns=lambda x: x.replace('ï»¿', '').strip(), inplace=True)
    # Jika ada kolom 'Invoice' kosong di ujung kanan akibat pergeseran delimiter, hapus juga:
    if 'Invoice' in df_clean.columns and df_clean.columns.duplicated().any():
        df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

    # Sort by date
    df_clean = df_clean.sort_values(by='InvoiceDate').reset_index(drop=True)
    
    print("\n" + "="*50)
    print("PIPELINE COMPLETE")
    print("="*50)
    print(f"Cleaned dataset shape: {df_clean.shape[0]:,} rows.")
    
    return df_clean

In [14]:
df_final = clean_retail_pipeline(df_raw)


Starting basic cleaning...

Starting advanced product standardization (Latest Transaction Method)...

PIPELINE COMPLETE
Cleaned dataset shape: 700,388 rows.


In [15]:
df_final.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085
4,489434,21232,STRAWBERRY CERAMIC TRINKET POT,24,2009-12-01 07:45:00,1.25,13085


In [16]:
df_final['StockCodeLen'] = df_final['StockCode'].str.len()

print("--- 1. Distribusi Panjang Karakter StockCode ---")
print(df_final['StockCodeLen'].value_counts())
print("-" * 50)

# Membuat kondisi filter menggunakan Regex
# Murni 5 digit angka
pola_standar = df_final['StockCode'].str.match(r'^\d{5}$') 

# 5 digit angka diikuti 1 sampai 2 huruf (Varian)
pola_varian = df_final['StockCode'].str.match(r'^\d{5}[a-zA-Z]{1,2}$') 

# Selain kedua pola di atas (Kode Administratif / Error)
pola_admin = ~(pola_standar | pola_varian)

print("\n--- 2. Sampel Kode Standar (5 Digit) ---")
print(df_final[pola_standar]['StockCode'].unique()[:5])

print("\n--- 3. Sampel Kode Varian (6-7 Karakter) ---")
print(df_final[pola_varian]['StockCode'].unique()[:10])

print("\n--- 4. Semua Kode Non-Standar ---")
print(df_final[pola_admin]['StockCode'].unique())

--- 1. Distribusi Panjang Karakter StockCode ---
StockCodeLen
5     620184
6      78531
7        918
1        599
2         58
4         52
12        30
3         16
Name: count, dtype: int64
--------------------------------------------------

--- 2. Sampel Kode Standar (5 Digit) ---
['85048' '22041' '21232' '22064' '21871']

--- 3. Sampel Kode Varian (6-7 Karakter) ---
['79323P' '79323W' '48173C' '84596F' '84596L' '35004B' '84970S' '84507B'
 '84031B' '84032A']

--- 4. Semua Kode Non-Standar ---
['M' 'POST' 'BANK CHARGES' 'C2' 'TEST001' 'TEST002' 'PADS' 'ADJUST' 'D'
 'ADJUST2' 'SP1002' 'DOT']


In [17]:
# Cek kode non-standar
lst = ['M','POST','BANK CHARGES','C2','TEST001','TEST002','PADS','ADJUST','D'
,'ADJUST2','SP1002','DOT']
df_check = df_final[df_final['StockCode'].isin(lst)]

# Lakukan grouping untuk membedah isi dari setiap StockCode
summary_check = df_check.groupby('StockCode').agg(
    Deskripsi_Unik=('Description', lambda x: list(x.dropna().unique())),
    Total_Baris=('Invoice', 'count'),
    Min_Quantity=('Quantity', 'min'),
    Max_Quantity=('Quantity', 'max'),
    Min_Price=('Price', 'min'),
    Max_Price=('Price', 'max')
).reset_index()

print("--- HASIL INVESTIGASI STOCKCODE NON-STANDAR ---")
print(summary_check.to_string(index=False))

--- HASIL INVESTIGASI STOCKCODE NON-STANDAR ---
   StockCode                        Deskripsi_Unik  Total_Baris  Min_Quantity  Max_Quantity  Min_Price  Max_Price
      ADJUST [ADJUSTMENT BY JOHN ON 26/01/2010 17]           10             1             1     15.580    387.540
     ADJUST2  [ADJUSTMENT BY PETER ON JUN 25 2010]            3             1             1     72.450    358.470
BANK CHARGES                        [BANK CHARGES]           30             1             1      0.001     15.000
          C2                            [CARRIAGE]           58             1             1     25.000     50.000
           D                            [DISCOUNT]            5             1           192      1.000    101.990
         DOT                      [DOTCOM POSTAGE]           16             1             1     11.170   1599.260
           M                              [MANUAL]          594             1          1600      0.060  10953.500
        PADS          [PADS TO MATCH ALL

### Penemuan: StockCode dengan Min_Price dan Max_Price yang sama merepresentasikan produk
StockCode
1. PADS --> tidak disimpan karena harga sangat kecil dan jumlah transaksi dan kuantitas yang sedikit (0.001 poundsterling)
2. SP1002
3. TEST001 --> tidak disimpan karena bukan produk beneran
4. TEST002 --> tidak disimpan karena bukan produk beneran

In [18]:
invalid_prefix = ('ADJUST', 'TEST')
df_final = df_final[~df_final['StockCode'].str.startswith(invalid_prefix)]

invalid_code = ['C2', 'POST', 'D', 'M', 'PADS', 'DOT', 'BANK CHARGES']
df_final = df_final[~df_final['StockCode'].astype(str).str.upper().isin(invalid_code)]
print(f"Final cleaned dataset shape: {df_final.shape[0]:,} rows.")

Final cleaned dataset shape: 699,610 rows.


In [19]:
display(df_final.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,StockCodeLen
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,5
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,6
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,6
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,5
4,489434,21232,STRAWBERRY CERAMIC TRINKET POT,24,2009-12-01 07:45:00,1.25,13085,5


In [20]:
df_final.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 699610 entries, 0 to 700387
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice       699610 non-null  object        
 1   StockCode     699610 non-null  object        
 2   Description   699610 non-null  object        
 3   Quantity      699610 non-null  int64         
 4   InvoiceDate   699610 non-null  datetime64[ns]
 5   Price         699610 non-null  float64       
 6   Customer ID   699610 non-null  object        
 7   StockCodeLen  699610 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 186.0 MB


In [21]:
# df_final.to_csv('/kaggle/working/cleaned_uk.csv', index=False)
df_final.to_csv('/kaggle/working/cleaned_uk_v2.csv', index=False)

# Deskripsi Statistik Dataset Transaksi

The performance of a mining algorithm primarily depends on the following two key factors:

1. Distribution of items’ frequencies and
2. Distribution of transaction length

Thus, it is important to know the statistical details of a database. PAMI provides inbuilt classes and functions methods to get the statistical details of a database. In this page, we provide the details of methods to get statistical details from a transactional database.

dari https://udaylab.github.io/PAMI/manuals/utilityDatabaseStats.html

In [22]:
# utk statistika dari library pami
!pip install pami==2026.1.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.3 MB/s eta 0:00:00
  Created wheel for JsonForm: filename=JsonForm-0.0.2-py3-none-any.whl size=3311 sha256=a4064c237df9d1f4945f362c97a1983be03ad9c3ee10f2b08672e533e1abc0c0
  Stored in directory: /root/.cache/pip/wheels/0b/29/3c/f5b5085becdbee0b282b60cda0028607f67adf8f099316a4a7
  Created wheel for JsonSir: filename=JsonSir-0.0.2-py3-none-any.whl size=4754 sha256=16fbbb44c1bf08f3108cdef5f5

In [23]:
# 1. Hitung nilai utility asal (Quantity * Price)
df_final['Utility'] = df_final['Quantity'] * df_final['Price']

# 2. Pastikan StockCode berupa string
df_final['StockCode'] = df_final['StockCode'].astype(str)

pami_output_file = '/kaggle/working/online_retail_pami.txt'
sep = '\t' 

print("Memulai konversi ke format PAMI (Nilai Utility dikalikan 100)...")

with open(pami_output_file, 'w', encoding='utf-8') as f:
    for invoice, group in df_final.groupby('Invoice'):
        items = group['StockCode'].tolist()
        
        # Kalikan setiap utility dengan 100 dan ubah menjadi integer bulat
        utilities = [int(round(u * 100)) for u in group['Utility']]
        total_utility = sum(utilities)
        
        items_str = sep.join(items)
        # Ubah list integer menjadi string yang dipisahkan TAB
        utilities_str = sep.join(str(u) for u in utilities)
        
        # Tulis ke file dengan format itemA\titemB:total_utility:utilA\tutilB
        pami_line = f"{items_str}:{total_utility}:{utilities_str}\n"
        f.write(pami_line)

print(f"Konversi selesai! File integer disimpan di: {pami_output_file}")

Memulai konversi ke format PAMI (Nilai Utility dikalikan 100)...
Konversi selesai! File integer disimpan di: /kaggle/working/online_retail_pami.txt


In [24]:
import PAMI.extras.dbStats.UtilityDatabase as uds

In [25]:
# Tentukan path file hasil konversi tadi
inputFile = '/kaggle/working/online_retail_pami.txt'
        
# PAMI default menggunakan tab separator, jadi tidak perlu di-override jika memakai tab
obj = uds.UtilityDatabase(inputFile)
obj.run()

# Statistika dataset setelah cleaning
print(f'Database size : {obj.getDatabaseSize()}')
print(f'Total number of items : {obj.getTotalNumberOfItems()}')
print(f'Database sparsity : {obj.getSparsity()}')  # Diubah dari printf ke print
print(f'Minimum Transaction Size : {obj.getMinimumTransactionLength()}')
print(f'Average Transaction Size : {obj.getAverageTransactionLength()}')
print(f'Maximum Transaction Size : {obj.getMaximumTransactionLength()}')
print(f'Standard Deviation Transaction Size : {obj.getStandardDeviationTransactionLength()}')
print(f'Variance in Transaction Sizes : {obj.getVarianceTransactionLength()}') # Diperbaiki kurung & spasi
print(f'Total utility : {obj.getTotalUtility()}')
print(f'Minimum utility : {obj.getMinimumUtility()}')
print(f'Average utility : {obj.getAverageUtility()}')
print(f'Maximum utility : {obj.getMaximumUtility()}')

# Menyimpan hasil distribusi statistik ke file csv
itemFrequencies = obj.getSortedListOfItemFrequencies()
transactionLength = obj.getTransanctionalLengthDistribution()
utility = obj.getSortedUtilityValuesOfItem()

Database size : 33361
Total number of items : 4605
Database sparsity : 0.9954460599005757
Minimum Transaction Size : 1
Average Transaction Size : 20.970894157848985
Maximum Transaction Size : 541
Standard Deviation Transaction Size : 23.00056674319779
Variance in Transaction Sizes : 529.041928603935
Total utility : 1428877350
Minimum utility : 38
Average utility : 310288.2410423453
Maximum utility : 22796551
